# Calibration Smoothing

**by Josh Dillon**, last updated August 28, 2026

This notebook smooths the per-file gains coming out of the
[file_sky_calibration](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/file_sky_calibration.ipynb)
notebook in time and frequency. It removes the flags found by that notebook and replaces them with
flags generated by
[full_day_rfi](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/full_day_rfi.ipynb)
and
[full_day_antenna_flagging](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/full_day_antenna_flagging.ipynb),
and it flags antennas with high relative difference between the original and smoothed gains.

Before smoothing, each file's fitted per-SNAP, per-X-engine-block decoherence staircase is removed
from its gains (using the `SNAPDecoherence` sidecars): the staircase is block-discontinuous
structure that the smooth basis would otherwise ring on, and it is re-applied downstream from the
sidecars themselves (e.g. by `apply_cal.calibrate_and_red_avg`), so **the smoothed gains written
here are staircase-free**. Gains on unmeasured decoherence blocks cannot be cleaned and are flagged
instead.

As the last stage that touches flags, this notebook also writes the day's complete a posteriori flag
yaml (entirely-flagged times, frequencies, and antennas).

Here's a set of links to skip to particular figures and tables:

• [Figure 1: Identifying and Blacklisting Sky-Cal Failures](#Figure-1:-Identifying-and-Blacklisting-Sky-Cal-Failures)

• [Figure 2: Antenna Phases with Identified Phase Flips](#Figure-2:-Antenna-Phases-with-Identified-Phase-Flips)

• [Figure 3: Full-Day Gain Amplitudes Before and After `smooth_cal`](#Figure-3:-Full-Day-Gain-Amplitudes-Before-and-After-smooth_cal)

• [Figure 4: Full-Day Gain Phases Before and After `smooth_cal`](#Figure-4:-Full-Day-Gain-Phases-Before-and-After-smooth_cal)

• [Figure 5: Full-Day $\chi^2$ / DoF Waterfall from Sky-Model Calibration](#Figure-5:-Full-Day-$\chi^2$-/-DoF-Waterfall-from-Sky-Model-Calibration)

• [Figure 6: Average $\chi^2$ per Antenna](#Figure-6:-Average-$\chi^2$-per-Antenna)

• [Figure 7: Relative Difference Before and After Smoothing](#Figure-7:-Relative-Difference-Before-and-After-Smoothing)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import glob
import copy
import warnings
import matplotlib
import matplotlib.pyplot as plt
from hera_cal import io, utils, smooth_cal
from hera_qm.time_series_metrics import true_stretches
%matplotlib inline
from IPython.display import display, HTML

## Parse inputs

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "CALIBRATION_SMOOTHING_NOTEBOOK")

# default suffixes for the files this notebook reads or writes
SUM_SUFFIX = 'sum.uvh5'
SKY_CAL_SUFFIX = 'sum.sky.calfits'
SMOOTH_CAL_SUFFIX = 'sum.smooth.calfits'
DECOHERENCE_SUFFIX = 'sum.snap_decoherence.h5'
ANTENNA_FLAGS_SUFFIX = 'sum.antenna_flags.h5'
FLAG_WATERFALL_SUFFIX = 'sum.flag_waterfall.h5'

# frequency dividing the low and high bands (inside the a priori flagged FM gap), shared
# across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz

# default settings, overridden by the TOML's [CALIBRATION_SMOOTHING_OPTS] section (if given)
FREQ_SMOOTHING_SCALE = 30.0  # in MHz
TIME_SMOOTHING_SCALE = 1e4  # in seconds
EIGENVAL_CUTOFF = 1e-12
PER_POL_REFANT = False
BANNED_REFANTS = [[144, 'Jnn'], [121, 'Jee'], [71, 'Jnn']]  # antennas never to use as reference antennas (H6C's list)
BLACKLIST_TIMESCALE_FACTOR = 4.0
BLACKLIST_RELATIVE_ERROR_THRESH = 1.0
BLACKLIST_RELATIVE_WEIGHT = 0.1
SC_RELATIVE_DIFF_CUTOFF = 0.2

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')

for toml_section in ['GLOBAL_OPTS', 'CALIBRATION_SMOOTHING_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

out_yaml_file = os.path.join(os.path.dirname(SUM_FILE), SUM_FILE.split('.')[-4] + '_aposteriori_flags.yaml')

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SUM_SUFFIX', 'SKY_CAL_SUFFIX',
                'SMOOTH_CAL_SUFFIX', 'DECOHERENCE_SUFFIX', 'ANTENNA_FLAGS_SUFFIX', 'FLAG_WATERFALL_SUFFIX',
                'BAND_SPLIT_FREQ', 'FREQ_SMOOTHING_SCALE', 'TIME_SMOOTHING_SCALE',
                'EIGENVAL_CUTOFF', 'PER_POL_REFANT', 'BANNED_REFANTS', 'BLACKLIST_TIMESCALE_FACTOR',
                'BLACKLIST_RELATIVE_ERROR_THRESH', 'BLACKLIST_RELATIVE_WEIGHT', 'SC_RELATIVE_DIFF_CUTOFF',
                'out_yaml_file']:
    print(f'{setting} = {eval(setting)}')

## Load files and select reference antenna(s)

In [ ]:
hd = io.HERAData(SUM_FILE)
sum_glob = '.'.join(SUM_FILE.split('.')[:-3]) + '.*.' + SUM_SUFFIX
cal_files_glob = sum_glob.replace(SUM_SUFFIX, SKY_CAL_SUFFIX)
cal_files = sorted(glob.glob(cal_files_glob))
print(f'Found {len(cal_files)} *.{SKY_CAL_SUFFIX} files starting with {cal_files[0]}.')

In [ ]:
rfi_flag_files_glob = sum_glob.replace(SUM_SUFFIX, FLAG_WATERFALL_SUFFIX)
rfi_flag_files = sorted(glob.glob(rfi_flag_files_glob))
print(f'Found {len(rfi_flag_files)} *.{FLAG_WATERFALL_SUFFIX} files starting with {rfi_flag_files[0]}.')

# every calibration file must have a matching flag waterfall: a shortfall means full_day_rfi
# ran before some per-file products existed and is stale relative to them
missing = sorted(set(cal.replace(SKY_CAL_SUFFIX, FLAG_WATERFALL_SUFFIX) for cal in cal_files) - set(rfi_flag_files))
assert len(missing) == 0, (f'{len(missing)} of {len(cal_files)} calibration files lack flag waterfalls '
                          f'(rerun full_day_rfi?), starting with {missing[0]}')

In [ ]:
ant_flag_files_glob = sum_glob.replace(SUM_SUFFIX, ANTENNA_FLAGS_SUFFIX)
ant_flag_files = sorted(glob.glob(ant_flag_files_glob))
print(f'Found {len(ant_flag_files)} *.{ANTENNA_FLAGS_SUFFIX} files starting with {ant_flag_files[0]}.')

# likewise, every calibration file must have a matching day-level antenna flags file
missing = sorted(set(cal.replace(SKY_CAL_SUFFIX, ANTENNA_FLAGS_SUFFIX) for cal in cal_files) - set(ant_flag_files))
assert len(missing) == 0, (f'{len(missing)} of {len(cal_files)} calibration files lack antenna flags '
                          f'(rerun full_day_antenna_flagging?), starting with {missing[0]}')

In [ ]:
cs = smooth_cal.CalibrationSmoother(cal_files, flag_file_list=(ant_flag_files + rfi_flag_files),
                                    ignore_calflags=True, pick_refant=False, load_chisq=True, load_cspa=True)

## Remove the fitted decoherence staircase

Each file's gains carry its fitted per-SNAP, per-X-engine-block suppression staircase —
block-discontinuous structure that the smooth DPSS basis would ring on. It is removed here
(`SNAPDecoherence.correct_gains`, using each sidecar's stored antenna → SNAP mapping, which includes
any identity repairs) so that the smoothed gains are staircase-free; downstream consumers apply
decoherence corrections from the sidecars themselves. Gains on unmeasured (`nan`) decoherence blocks
cannot be cleaned, so they are flagged rather than allowed to bias the smooth fits.

In [ ]:
deco_files = [cal.replace(SKY_CAL_SUFFIX, DECOHERENCE_SUFFIX) for cal in cal_files]
n_flagged = 0
for cal, deco_file in zip(cal_files, deco_files):
    sd = io.SNAPDecoherence.read(deco_file)
    inds = cs.time_indices[cal]
    mapped = [ant for ant in cs.gain_grids if ant[0] in sd.ant_to_SNAP_dict]
    corrected = sd.correct_gains({ant: cs.gain_grids[ant][inds] for ant in mapped})
    nchans_per_block = sd.block_freqs.shape[1]
    unmeasured = {SNAP: np.repeat(np.isnan(p), nchans_per_block, axis=1) for SNAP, p in sd.decoherence.items()}
    for ant in mapped:
        cs.gain_grids[ant][inds] = corrected[ant]
        if sd.ant_to_SNAP_dict[ant[0]] in unmeasured:
            new_flags = unmeasured[sd.ant_to_SNAP_dict[ant[0]]] & ~cs.flag_grids[ant][inds]
            n_flagged += np.sum(new_flags)
            cs.flag_grids[ant][inds] |= new_flags
print(f'Removed decoherence staircases from {len(cal_files)} files\' gains; '
      f'flagged {n_flagged} previously-unflagged (ant, time, channel) cells on unmeasured blocks.')

In [ ]:
# Pick reference antenna(s) but don't let ants known to flip phases get picked as reference antennas
banned_refants = [tuple(ant) for ant in BANNED_REFANTS]
cs.refant = smooth_cal.pick_reference_antenna({ant: cs.gain_grids[ant] for ant in cs.gain_grids if ant not in banned_refants},
                                              {ant: cs.flag_grids[ant] for ant in cs.gain_grids if ant not in banned_refants},
                                              cs.freqs, per_pol=True, acceptable_candidate_frac=0.25, antpos=hd.antpos)
for pol in cs.refant:
    print(f'Reference antenna {cs.refant[pol][0]} selected for smoothing {pol} gains.')

if not PER_POL_REFANT:
    # in this case, rephase both pols separately before smoothing, but also smooth the relative polarization calibration phasor
    overall_refant = smooth_cal.pick_reference_antenna({ant: cs.gain_grids[ant] for ant in cs.refant.values()}, 
                                                       {ant: cs.flag_grids[ant] for ant in cs.refant.values()}, 
                                                       cs.freqs, per_pol=False)
    print(f'Overall reference antenna {overall_refant} selected.')
    other_refant = [ant for ant in cs.refant.values() if ant != overall_refant][0]

    relative_pol_phasor = cs.gain_grids[overall_refant] * cs.gain_grids[other_refant].conj() # TODO: is this conjugation right?
    relative_pol_phasor /= np.abs(relative_pol_phasor)

sky_cal_refants = {cs.refant[pol]: cs.gain_grids[cs.refant[pol]] for pol in ['Jee', 'Jnn']}

In [ ]:
cs.rephase_to_refant(propagate_refant_flags=True)

In [ ]:
lst_grid = utils.JD2LST(cs.time_grid) * 12 / np.pi
lst_grid[lst_grid > lst_grid[-1]] -= 24
frac_jds = cs.time_grid - int(cs.time_grid[0])

## Find consistent outliers in relative error after a coarse smoothing
These are typically a sign of failures of the per-file sky calibration.

In [ ]:
grid_shape = next(iter(cs.gain_grids.values())).shape
relative_error_samples = {pol: np.zeros(grid_shape) for pol in ['Jee', 'Jnn']}
sum_relative_error = {pol: np.zeros(grid_shape) for pol in ['Jee', 'Jnn']}
per_ant_avg_relative_error = {} 

# perform a 2D DPSS filter with a BLACKLIST_TIMESCALE_FACTOR longer timescale, averaging the results per-pol
for ant in cs.gain_grids:
    if np.all(cs.flag_grids[ant]):
        continue
    filtered, _ = smooth_cal.time_freq_2D_filter(gains=cs.gain_grids[ant], 
                                                 wgts=(~cs.flag_grids[ant]).astype(float),
                                                 freqs=cs.freqs,
                                                 times=cs.time_grid,
                                                 freq_scale=FREQ_SMOOTHING_SCALE,
                                                 time_scale=TIME_SMOOTHING_SCALE * BLACKLIST_TIMESCALE_FACTOR,
                                                 eigenval_cutoff=EIGENVAL_CUTOFF,
                                                 method='DPSS', 
                                                 fit_method='lu_solve', 
                                                 fix_phase_flips=True, 
                                                 phase_flip_time_scale = TIME_SMOOTHING_SCALE / 2,
                                                 flag_phase_flip_ints=True,
                                                 skip_flagged_edges=True, 
                                                 freq_cuts=[BAND_SPLIT_FREQ * 1e6],
                                                ) 
    relative_error = np.where(cs.flag_grids[ant], 0, np.abs(cs.gain_grids[ant] - filtered) / np.abs(filtered))
    per_ant_avg_relative_error[ant] = np.nanmean(np.where(cs.flag_grids[ant], np.nan, relative_error))
    relative_error_samples[ant[1]] += (~cs.flag_grids[ant]).astype(float)
    sum_relative_error[ant[1]] += relative_error

# figure out per-antpol cuts for where to set weights to 0 for the main smooth_cal (but not necessarily flags)
cs.blacklist_wgt = BLACKLIST_RELATIVE_WEIGHT
for pol in ['Jee', 'Jnn']:
    avg_rel_error = sum_relative_error[pol] / relative_error_samples[pol]
    to_blacklist = np.where(relative_error_samples[pol] > 0, avg_rel_error > BLACKLIST_RELATIVE_ERROR_THRESH, False)
    for ant in cs.ants:
        if ant[1] == pol:
            cs.waterfall_blacklist[ant] = to_blacklist

In [ ]:
def plot_relative_error():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        fig, axes = plt.subplots(1, 3, figsize=(14, 6), dpi=150, constrained_layout=True)
        extent = [cs.freqs[0] / 1e6, cs.freqs[-1] / 1e6, frac_jds[-1], frac_jds[0]]
        cmap = plt.get_cmap('Greys', 256)
        cmap.set_over('red')
        for ax, pol in zip(axes[0:2], ['Jee', 'Jnn']):
            to_plot = sum_relative_error[pol] / relative_error_samples[pol]
            im = ax.imshow(np.where(np.isfinite(to_plot), to_plot, np.nan), aspect='auto', interpolation='none', 
                           vmin=0, vmax=BLACKLIST_RELATIVE_ERROR_THRESH, extent=extent, cmap=cmap)
            ax.set_title(pol)
            ax.set_xlabel('Frequency (MHz)')
        axes[0].set_ylabel(f'JD - {int(cs.time_grid[0])}')
        fig.colorbar(im, ax=list(axes[0:2]), location='top', extend='max', aspect=40,
                     label='Average Relative Error on Initial Smoothing')

        # Add LST right axis on the rightmost waterfall panel
        ax2 = axes[1].twinx()
        ax2.set_ylim(lst_grid[-1], lst_grid[0])
        mod24 = lambda x, _: f"{x % 24:.1f}"
        ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
        ax2.set_ylabel('LST (hours)')
        
        for pol in ['Jee', 'Jnn']:
            axes[2].hist((sum_relative_error[pol] / relative_error_samples[pol]).ravel(), bins=np.arange(0, 2, .01), alpha=.5, label=pol)
        axes[2].set_yscale('log')
        axes[2].set_ylabel('Number of Waterfall Pixels')
        axes[2].set_xlabel('Relative Error')
        axes[2].axvline(BLACKLIST_RELATIVE_ERROR_THRESH, ls='--', c='r', label='Blacklist Threshold')
        axes[2].legend()

# *Figure 1: Identifying and Blacklisting Sky-Cal Failures*

This plot highlights regions of the waterfall that are per-polarization blacklisted (i.e. given 0
weight in the main `smooth_cal` fit, but not necessarily flagged). This is usually a sign of
problems with sky-cal and often occurs because

In [ ]:
plot_relative_error()

In [ ]:
# duplicate a small number of sky-cal gains for plotting
antnums = set([ant[0] for ant in cs.ants])
flags_per_antnum = [np.sum(cs.flag_grids[ant, 'Jnn']) + np.sum(cs.flag_grids[ant, 'Jee']) for ant in antnums]
larger_relative_error = np.array([np.max([per_ant_avg_relative_error.get((ant, pol), np.inf) for pol in ['Jee', 'Jnn']]) for ant in antnums])
refant_nums = [ant[0] for ant in cs.refant.values()]

# pick candidates that don't exhibit too many flags or non-smooth structure on first pass
candidate_ants = []
rel_error_factor = 1
while len(candidate_ants) < 6:  # Select more candidates to ensure we have enough after potential flagging
    candidate_ants = [ant for ant, nflags, rel_err in zip(antnums, flags_per_antnum, larger_relative_error) 
                      if (ant not in refant_nums) and (nflags <= np.percentile(flags_per_antnum, 25))
                      and (rel_err <= SC_RELATIVE_DIFF_CUTOFF * rel_error_factor)
                      and not np.all(cs.flag_grids[ant, 'Jee']) and not np.all(cs.flag_grids[ant, 'Jnn'])]
    rel_error_factor += .1

# choose antennas to plot: select up to 6 candidates, prioritizing diversity across antenna numbers
candidate_ants_sorted = sorted(candidate_ants)
step = max(1, len(candidate_ants_sorted) // 6)  # spread them out
_candidates = sorted(candidate_ants_sorted[::step][:6])
ants_to_plot_candidates = [_candidates[i//2] if i % 2 == 0 else _candidates[-(i//2)-1] for i in range(len(_candidates))]

# Store sky-cal gains for all candidates
sky_cal_gains = {}
for pol in ['Jee', 'Jnn']:
    for antnum in ants_to_plot_candidates:
        if PER_POL_REFANT:
            sky_cal_gains[antnum, pol] = cs.gain_grids[(antnum, pol)] * np.abs(sky_cal_refants[cs.refant[pol]]) / sky_cal_refants[cs.refant[pol]]
        else:
            sky_cal_gains[antnum, pol] = cs.gain_grids[(antnum, pol)] / np.abs(sky_cal_refants[cs.refant[pol]]) * sky_cal_refants[cs.refant[pol]]
            sky_cal_gains[antnum, pol] *= np.abs(sky_cal_refants[overall_refant]) / sky_cal_refants[overall_refant]

## Perform smoothing

In [ ]:
if not PER_POL_REFANT:
    # treat the relative_pol_phasor as if it were antenna -1
    cs.gain_grids[(-1, other_refant[1])] = relative_pol_phasor
    cs.flag_grids[(-1, other_refant[1])] = cs.flag_grids[overall_refant] | cs.flag_grids[other_refant]
    cs.waterfall_blacklist[(-1, other_refant[1])] = cs.waterfall_blacklist[cs.ants[0][0], 'Jee'] | cs.waterfall_blacklist[cs.ants[0][0], 'Jnn'] 

In [ ]:
meta = cs.time_freq_2D_filter(freq_scale=FREQ_SMOOTHING_SCALE,
                              time_scale=TIME_SMOOTHING_SCALE,
                              eigenval_cutoff=EIGENVAL_CUTOFF,
                              method='DPSS', 
                              fit_method='lu_solve',
                              fix_phase_flips=True,
                              phase_flip_time_scale = TIME_SMOOTHING_SCALE / 2,
                              flag_phase_flip_ints=True,
                              skip_flagged_edges=True,
                              freq_cuts=[BAND_SPLIT_FREQ * 1e6],)

In [ ]:
# calculate average chi^2 per antenna before additional flagging
avg_cspa_vs_time = {ant: np.nanmean(np.where(cs.flag_grids[ant], np.nan, cs.cspa_grids[ant]), axis=1) for ant in cs.ants}
avg_cspa_vs_freq = {ant: np.nanmean(np.where(cs.flag_grids[ant], np.nan, cs.cspa_grids[ant]), axis=0) for ant in cs.ants}
avg_cspa = {ant: np.nanmean(np.where(cs.flag_grids[ant], np.nan, cs.cspa_grids[ant])) for ant in cs.ants}

In [ ]:
# Pick out antennas with too high relative differences before and after smoothing and flag them.
avg_relative_diffs = {ant: np.nanmean(rel_diff) for ant, rel_diff in meta['freq_avg_rel_diff'].items()}
to_cut = sorted([ant for ant, diff in avg_relative_diffs.items() if ant[0] >= 0 and diff > SC_RELATIVE_DIFF_CUTOFF])
if len(to_cut) > 0:
    for ant in to_cut:
        print(f'Flagging antenna {ant[0]}{ant[1][-1]} with a relative difference before and after smoothing of {avg_relative_diffs[ant]:.2%} '
              f'(compared to the {SC_RELATIVE_DIFF_CUTOFF:.2%} cutoff).')
        cs.flag_grids[ant] |= True
else:
    print(f'No antennas have a relative difference above the {SC_RELATIVE_DIFF_CUTOFF:.2%} cutoff.')

In [ ]:
if not PER_POL_REFANT:
    # put back in the smoothed phasor, ensuring the amplitude is 1 and that data are flagged anywhere either polarization's refant is flagged
    smoothed_relative_pol_phasor = cs.gain_grids[(-1, other_refant[-1])] / np.abs(cs.gain_grids[(-1, other_refant[-1])])
    for ant in cs.gain_grids:
        if ant[0] >= 0 and ant[1] == other_refant[1]:
            cs.gain_grids[ant] /= smoothed_relative_pol_phasor
        cs.flag_grids[ant] |= (cs.flag_grids[(-1, other_refant[1])])
    cs.refant = overall_refant

In [ ]:
def phase_flip_diagnostic_plot():
    '''Shows time-smoothed antenna avg phases after taking out a delay and filtering in time.'''
    if not np.any([np.any(meta['phase_flipped'][ant]) for ant in meta['phase_flipped']]):
        print("No antennas have phase flips identified. Nothing to plot.")
        return
    
    plt.figure(figsize=(14,4), dpi=150)
    for ant in meta['phase_flipped']:
        if np.any(meta['phase_flipped'][ant]):
            to_plot = np.angle(np.exp(1.0j * (meta['phases'][ant] - meta['time_smoothed_phases'][ant])))
            to_plot[to_plot < -np.pi / 2] += 2 * np.pi
            plt.plot(cs.time_grid - int(cs.time_grid[0]), to_plot, label=f'{ant[0]}{ant[1][-1]}')
    plt.legend(title='Antennas with Identified Phase Flips', ncol=4)
    plt.xlabel(f'JD - {int(cs.time_grid[0])}')
    plt.ylabel('Average Phase After Filtering (radians)')
    plt.tight_layout()

# *Figure 2: Antenna Phases with Identified Phase Flips*

In [ ]:
phase_flip_diagnostic_plot()

## Plot results

In [ ]:
def amplitude_plot(ant_to_plot):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Pick vmax to not saturate 90% of the sky-cal gains
        vmax = np.max([np.percentile(np.abs(cs.gain_grids[ant_to_plot, pol][~cs.flag_grids[ant_to_plot, pol]]), 99) for pol in ['Jee', 'Jnn']])

        display(HTML(f'<h2>Antenna {ant_to_plot} Amplitude Waterfalls</h2>'))    

        # Plot sky-cal gain amplitude waterfalls for a single antenna
        fig, axes = plt.subplots(4, 2, figsize=(14,14), dpi=150, gridspec_kw={'height_ratios': [1, 1, .4, .4]})
        for ax, pol in zip(axes[0], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            extent=[cs.freqs[0]/1e6, cs.freqs[-1]/1e6, frac_jds[-1], frac_jds[0]]
            im = ax.imshow(np.where(cs.flag_grids[ant], np.nan, np.abs(cs.gain_grids[ant])), aspect='auto', cmap='inferno', 
                           interpolation='nearest', vmin=0, vmax=vmax, extent=extent)
            ax.set_title(f'Smoothcal Gain Amplitude of Antenna {ant[0]}: {pol[-1]}-polarized' )
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel(f'JD - {int(cs.time_grid[0])}')
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])
            plt.colorbar(im, ax=ax,  orientation='horizontal', pad=.15)

        # Now flagged plot sky-cal waterfall    
        for ax, pol in zip(axes[1], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            extent=[cs.freqs[0]/1e6, cs.freqs[-1]/1e6, frac_jds[-1], frac_jds[0]]
            im = ax.imshow(np.where(cs.flag_grids[ant], np.nan, np.abs(sky_cal_gains[ant])), aspect='auto', cmap='inferno', 
                           interpolation='nearest', vmin=0, vmax=vmax, extent=extent)
            ax.set_title(f'Sky-Cal Gain Amplitude of Antenna {ant[0]}: {pol[-1]}-polarized' )
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel(f'JD - {int(cs.time_grid[0])}')
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])
            plt.colorbar(im, ax=ax,  orientation='horizontal', pad=.15)

        # Add LST right axis to rightmost waterfall columns
        for row in [0, 1]:
            ax2 = axes[row, 1].twinx()
            ax2.set_ylim(lst_grid[-1], lst_grid[0])
            mod24 = lambda x, _: f"{x % 24:.1f}"
            ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
            ax2.set_ylabel('LST (hours)')

        # Now plot mean gain spectra 
        for ax, pol in zip(axes[2], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)   
            nflags_spectrum = np.sum(cs.flag_grids[ant], axis=0)
            to_plot = nflags_spectrum <= np.percentile(nflags_spectrum, 75)
            ax.plot(cs.freqs[to_plot] / 1e6, np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.abs(sky_cal_gains[ant])), axis=0)[to_plot], 'r.', label='Sky-Cal')        
            ax.plot(cs.freqs[to_plot] / 1e6, np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.abs(cs.gain_grids[ant])), axis=0)[to_plot], 'k.', ms=2, label='Smoothed')        
            ax.set_ylim([0, vmax])
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])    
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel('|g| (unitless)')
            ax.set_title(f'Mean Infrequently-Flagged Gain Amplitude of Antenna {ant[0]}: {pol[-1]}-polarized')
            ax.legend(loc='upper left')

        # Now plot mean gain time series
        for ax, pol in zip(axes[3], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            nflags_series = np.sum(cs.flag_grids[ant], axis=1)
            to_plot = nflags_series <= np.percentile(nflags_series, 75)
            ax.plot(lst_grid[to_plot], np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.abs(sky_cal_gains[ant])), axis=1)[to_plot], 'r.', label='Sky-Cal')        
            ax.plot(lst_grid[to_plot], np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.abs(cs.gain_grids[ant])), axis=1)[to_plot], 'k.', ms=2, label='Smoothed')        
            ax.set_ylim([0, vmax])
            ax.set_xlabel('LST (hours)')
            ax.set_ylabel('|g| (unitless)')
            ax.set_title(f'Mean Infrequently-Flagged Gain Amplitude of Antenna {ant[0]}: {pol[-1]}-polarized')
            ax.set_xticklabels(ax.get_xticks() % 24)
            ax.legend(loc='upper left')

        plt.tight_layout()
        plt.show()    

In [ ]:
def phase_plot(ant_to_plot):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")    
        display(HTML(f'<h2>Antenna {ant_to_plot} Phase Waterfalls</h2>'))
        fig, axes = plt.subplots(4, 2, figsize=(14,14), dpi=150, gridspec_kw={'height_ratios': [1, 1, .4, .4]})
        
        # Plot phase waterfalls for a single antenna    
        for ax, pol in zip(axes[0], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            extent=[cs.freqs[0]/1e6, cs.freqs[-1]/1e6, frac_jds[-1], frac_jds[0]]
            im = ax.imshow(np.where(cs.flag_grids[ant], np.nan, np.angle(cs.gain_grids[ant])), aspect='auto', cmap='inferno', 
                           interpolation='nearest', vmin=-np.pi, vmax=np.pi, extent=extent)

            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_title(f'Smoothcal Gain Phase of Ant {ant[0]}{pol[-1]} / Ant {refant[0]}{refant[1][-1]}')
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel(f'JD - {int(cs.time_grid[0])}')
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])
            plt.colorbar(im, ax=ax,  orientation='horizontal', pad=.15)

        # Now plot sky-cal phase waterfall    
        for ax, pol in zip(axes[1], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            extent=[cs.freqs[0]/1e6, cs.freqs[-1]/1e6, frac_jds[-1], frac_jds[0]]
            im = ax.imshow(np.where(cs.flag_grids[ant], np.nan, np.angle(sky_cal_gains[ant])), aspect='auto', cmap='inferno', 
                           interpolation='nearest', vmin=-np.pi, vmax=np.pi, extent=extent)
            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_title(f'Sky-Cal Gain Phase of Ant {ant[0]}{pol[-1]} / Ant {refant[0]}{refant[1][-1]}')
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel(f'JD - {int(cs.time_grid[0])}')
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])
            plt.colorbar(im, ax=ax,  orientation='horizontal', pad=.15)

        # Add LST right axis to rightmost waterfall columns
        for row in [0, 1]:
            ax2 = axes[row, 1].twinx()
            ax2.set_ylim(lst_grid[-1], lst_grid[0])
            mod24 = lambda x, _: f"{x % 24:.1f}"
            ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
            ax2.set_ylabel('LST (hours)')

        # Now plot median gain spectra 
        for ax, pol in zip(axes[2], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)   
            nflags_spectrum = np.sum(cs.flag_grids[ant], axis=0)
            to_plot = nflags_spectrum <= np.percentile(nflags_spectrum, 75)
            ax.plot(cs.freqs[to_plot] / 1e6, np.nanmedian(np.where(cs.flag_grids[ant], np.nan, np.angle(sky_cal_gains[ant])), axis=0)[to_plot], 'r.', label='Sky-Cal')        
            ax.plot(cs.freqs[to_plot] / 1e6, np.nanmedian(np.where(cs.flag_grids[ant], np.nan, np.angle(cs.gain_grids[ant])), axis=0)[to_plot], 'k.', ms=2, label='Smoothed')        
            ax.set_ylim([-np.pi, np.pi])
            ax.set_xlim([cs.freqs[0]/1e6, cs.freqs[-1]/1e6])    
            ax.set_xlabel('Frequency (MHz)')
            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_ylabel(f'Phase of g$_{{{ant[0]}{pol[-1]}}}$ / g$_{{{refant[0]}{refant[1][-1]}}}$')
            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_title(f'Median Infrequently-Flagged Gain Phase of Ant {ant[0]}{pol[-1]} / Ant {refant[0]}{refant[1][-1]}')
            ax.legend(loc='upper left')

        # # Now plot median gain time series
        for ax, pol in zip(axes[3], ['Jee', 'Jnn']):
            ant = (ant_to_plot, pol)
            nflags_series = np.sum(cs.flag_grids[ant], axis=1)
            to_plot = nflags_series <= np.percentile(nflags_series, 75)
            ax.plot(lst_grid[to_plot], np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.angle(sky_cal_gains[ant])), axis=1)[to_plot], 'r.', label='Sky-Cal')        
            ax.plot(lst_grid[to_plot], np.nanmean(np.where(cs.flag_grids[ant], np.nan, np.angle(cs.gain_grids[ant])), axis=1)[to_plot], 'k.', ms=2, label='Smoothed')        
            ax.set_ylim([-np.pi, np.pi])    
            ax.set_xlabel('LST (hours)')
            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_ylabel(f'Phase of g$_{{{ant[0]}{pol[-1]}}}$ / g$_{{{refant[0]}{refant[1][-1]}}}$')
            refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
            ax.set_title(f'Mean Infrequently-Flagged Gain Phase of Ant {ant[0]}{pol[-1]} / Ant {refant[0]}{refant[1][-1]}')
            ax.set_xticklabels(ax.get_xticks() % 24)    
            ax.legend(loc='upper left')

        plt.tight_layout()
        plt.show()

In [ ]:
# Select first 2 unflagged antennas from candidates for amplitude plotting
ants_to_plot = []
for ant_candidate in ants_to_plot_candidates:
    if not (np.all(cs.flag_grids[ant_candidate, 'Jee']) and np.all(cs.flag_grids[ant_candidate, 'Jnn'])):
        ants_to_plot.append(ant_candidate)
        if len(ants_to_plot) >= 2:
            break

# *Figure 3: Full-Day Gain Amplitudes Before and After `smooth_cal`*

Here we plot sky-cal and `smooth_cal` gain amplitudes for both of the sample antennas. We also show
means across time/frequency, excluding frequencies/times that are frequently flagged.

In [ ]:
if len(ants_to_plot) == 0:
    print("Warning: No unflagged antennas available for plotting.")
else:
    for ant_to_plot in ants_to_plot:
        amplitude_plot(ant_to_plot)

# *Figure 4: Full-Day Gain Phases Before and After `smooth_cal`*

Here we plot sky-cal and `smooth_cal` phases relative to each polarization's reference antenna for
both of the sample antennas. We also show medians across time/frequency, excluding frequencies/times
that are frequently flagged.

In [ ]:
# Use the same selected unflagged antennas for phase plotting
if len(ants_to_plot) == 0:
    print("Warning: No unflagged antennas available for plotting.")
else:
    for ant_to_plot in ants_to_plot:
        phase_plot(ant_to_plot)

## Examine $\chi^2$

In [ ]:
def chisq_plot():
    fig, axes = plt.subplots(1, 2, figsize=(14, 10), dpi=150, sharex=True, sharey=True)
    extent = [cs.freqs[0]/1e6, cs.freqs[-1]/1e6, frac_jds[-1], frac_jds[0]]
    for ax, pol in zip(axes, ['Jee', 'Jnn']):
        refant = (cs.refant[pol] if isinstance(cs.refant, dict) else cs.refant)
        im = ax.imshow(np.where(cs.flag_grids[refant], np.nan, cs.chisq_grids[pol]), vmin=1, vmax=5, 
                       aspect='auto', cmap='turbo', interpolation='none', extent=extent)
        ax.set_title(f'{pol[1:]}-Polarized $\\chi^2$ / DoF')
        ax.set_xlabel('Frequency (MHz)')

    axes[0].set_ylabel(f'JD - {int(cs.time_grid[0])}')

    # Add LST right axis
    ax2 = axes[1].twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()
    fig.colorbar(im, ax=axes, pad=.07, label='$\\chi^2$ / DoF', orientation='horizontal', extend='both', aspect=50)

# *Figure 5: Full-Day $\chi^2$ / DoF Waterfall from Sky-Model Calibration*

Here we plot $\chi^2$ per degree of freedom from redundant-baseline calibration for both polarizations separately. While this plot is a little out of place, as it was not produced by this notebook, it is a convenient place where all the necessary components are readily available. If the array were perfectly redundant and any non-redundancies in the calibrated visibilities were explicable by thermal noise alone, this waterfall should be all 1.  

In [ ]:
chisq_plot()

In [ ]:
def cspa_vs_time_plot():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), dpi=150, sharex=True, sharey=True, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['Jee', 'Jnn']):
        detail_cutoff = np.percentile([np.nanmean(m) for ant, m in avg_cspa_vs_time.items() 
                                       if ant[1] == pol and np.isfinite(np.nanmean(m))], 95)
        for ant in avg_cspa_vs_time:
            if ant[1] == pol and not np.all(cs.flag_grids[ant]):
                if np.nanmean(avg_cspa_vs_time[ant]) > detail_cutoff:
                    ax.plot(lst_grid, avg_cspa_vs_time[ant], label=str((int(ant[0]), ant[1])), zorder=100)
                else:
                    ax.plot(lst_grid, avg_cspa_vs_time[ant], c='grey', alpha=.2, lw=.5)
        ax.legend(title=f'{pol[1:]}-Polarized', ncol=2)
        ax.set_ylabel('Mean Unflagged $\\chi^2$ per Antenna')
        ax.set_xlabel('LST (hours)')
        ax.set_xticklabels(ax.get_xticks() % 24)

    plt.ylim([1, 5.4])
    plt.tight_layout()

In [ ]:
def cspa_vs_freq_plot():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), dpi=150, sharex=True, sharey=True, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['Jee', 'Jnn']):
        detail_cutoff = np.percentile([np.nanmean(m) for ant, m in avg_cspa_vs_freq.items() 
                                       if ant[1] == pol and np.isfinite(np.nanmean(m))], 95)
        for ant in avg_cspa_vs_freq:
            if ant[1] == pol and not np.all(cs.flag_grids[ant]):
                if np.nanmean(avg_cspa_vs_freq[ant]) > detail_cutoff:
                    ax.plot(cs.freqs / 1e6, avg_cspa_vs_freq[ant], label=str((int(ant[0]), ant[1])), zorder=100)
                else:
                    ax.plot(cs.freqs / 1e6, avg_cspa_vs_freq[ant], c='grey', alpha=.2, lw=.5)
        ax.legend(title=f'{pol[1:]}-Polarized', ncol=2)
        ax.set_ylabel('Mean Unflagged $\\chi^2$ per Antenna')
        ax.set_xlabel('Frequency (MHz)')

    plt.ylim([1, 5.4])
    plt.tight_layout()

In [ ]:
def avg_cspa_array_plot():
    hd = io.HERAData(SUM_FILE)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=150, sharex=True, sharey=True, gridspec_kw={'wspace': 0})
    for pol, ax in zip(['Jee', 'Jnn'], axes):

        ants_here = [ant for ant in avg_cspa if np.isfinite(avg_cspa[ant]) and ant[1] == pol if ant[0] in hd.antpos]
        avg_chisqs = [avg_cspa[ant] for ant in ants_here]
        xs = [hd.antpos[ant[0]][0] for ant in ants_here]
        ys = [hd.antpos[ant[0]][1] for ant in ants_here]
        names = [ant[0] for ant in ants_here]
        
        im = ax.scatter(x=xs, y=ys, c=avg_chisqs, s=200, vmin=1, vmax=3, cmap='turbo')
        ax.set_aspect('equal')
        for x,y,n in zip(xs, ys, names):
            ax.text(x, y, str(n), va='center', ha='center', fontsize=8)
        ax.set_title(pol)
        ax.set_xlabel('East-West Antenna Position (m)')
    
    axes[0].set_ylabel('North-South Antenna Position (m)')

    plt.tight_layout()
    plt.colorbar(im, ax=axes, location='top', aspect=60, pad=.04, label='Mean Unflagged $\\chi^2$ per Antenna', extend='both')

# *Figure 6: Average $\chi^2$ per Antenna*

Here we plot $\chi^2$ per antenna from redundant-baseline calibration, separating polarizations and
averaging the unflagged pixels in the waterfalls over frequency or time. The worst 5% of antennas
are shown in color and highlighted in the legends, the rest are shown in grey. We also show time-
and frequency-averaged $\chi^2$ for each antennas as a scatter plot with array position.

In [ ]:
cspa_vs_freq_plot()
cspa_vs_time_plot()
avg_cspa_array_plot()

## Examine relative differences before and after smoothing

In [ ]:
def time_avg_diff_plot():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), dpi=150, sharex=True, sharey=True, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['Jee', 'Jnn']):
        detail_cutoff = np.percentile([np.nanmean(diff) for ant, diff in meta['time_avg_rel_diff'].items() 
                                       if ant[1] == pol and np.isfinite(np.nanmean(diff))], 95)    
        for ant, rel_diff in meta['time_avg_rel_diff'].items():
            if ant[0] >= 0 and ant[1] == pol and np.any(np.isfinite(rel_diff)):
                if np.nanmean(rel_diff) > detail_cutoff:
                    if np.all(cs.flag_grids[ant]):
                        ax.plot(cs.freqs / 1e6, rel_diff, label=str((int(ant[0]), ant[1])), zorder=99, ls='--', c='r', lw=.5)    
                    else:
                        ax.plot(cs.freqs / 1e6, rel_diff, label=str((int(ant[0]), ant[1])), zorder=100)
                else:
                    ax.plot(cs.freqs / 1e6, rel_diff, c='grey', alpha=.2, lw=.5)
        med_rel_diff = np.nanmedian([diff for ant, diff in meta['time_avg_rel_diff'].items() if ant[1] == pol], axis=0)
        ax.plot(cs.freqs / 1e6, med_rel_diff, 'k--', label='Median')
        ax.set_ylim([0, 1.05])
        ax.legend(title=f'{pol[1:]}-Polarized', ncol=2)
        ax.set_ylabel('Time-Averaged Relative Difference\nBefore and After Smoothing')
        ax.set_xlabel('Frequency (MHz)')
    plt.tight_layout()

In [ ]:
def freq_avg_diff_plot():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), dpi=150, sharex=True, sharey=True, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['Jee', 'Jnn']):
        detail_cutoff = np.percentile([np.nanmean(m) for ant, m in meta['freq_avg_rel_diff'].items() 
                                       if ant[1] == pol and np.isfinite(np.nanmean(m))], 95)    
        for ant, rel_diff in meta['freq_avg_rel_diff'].items():
            if ant[0] >= 0 and ant[1] == pol and np.any(np.isfinite(rel_diff)):
                if np.nanmean(rel_diff) > detail_cutoff:
                    if np.all(cs.flag_grids[ant]):
                        ax.plot(lst_grid, rel_diff, label=str((int(ant[0]), ant[1])), zorder=99, ls='--', c='r', lw=.5)    
                    else:
                        ax.plot(lst_grid, rel_diff, label=str((int(ant[0]), ant[1])), zorder=100)
                else:
                    ax.plot(lst_grid, rel_diff, c='grey', alpha=.2, lw=.5)
        
        med_rel_diff = np.nanmedian([diff for ant, diff in meta['freq_avg_rel_diff'].items() if ant[1] == pol], axis=0)
        ax.plot(lst_grid, med_rel_diff, 'k--', label='Median', zorder=101)
        ax.set_ylim([0, 1.05])
        ax.legend(title=f'{pol[1:]}-Polarized', ncol=2)
        ax.set_ylabel('Frequency-Averaged Relative Difference\nBefore and After Smoothing')
        ax.set_xlabel('LST (hours)')
        ax.set_xticklabels(ax.get_xticks() % 24)
    plt.tight_layout()

In [ ]:
def avg_difference_array_plot():
    hd = io.HERAData(SUM_FILE)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), dpi=150, sharex=True, sharey=True, gridspec_kw={'wspace': 0})
    for pol, ax in zip(['Jee', 'Jnn'], axes):
    
        avg_diffs = [np.nanmean(meta['time_avg_rel_diff'][ant]) for ant in meta['time_avg_rel_diff'] if ant[1] == pol if ant[0] in hd.antpos]
        xs = [hd.antpos[ant[0]][0] for ant in meta['time_avg_rel_diff'] if ant[1] == pol if ant[0] in hd.antpos]
        ys = [hd.antpos[ant[0]][1] for ant in meta['time_avg_rel_diff'] if ant[1] == pol if ant[0] in hd.antpos]
        names = [ant[0] for ant in meta['time_avg_rel_diff'] if ant[1] == pol if ant[0] in hd.antpos]
        
        im = ax.scatter(x=xs, y=ys, c=avg_diffs, s=200, vmin=0, vmax=.25, cmap='turbo')
        ax.set_aspect('equal')
        for x,y,n in zip(xs, ys, names):
            color = ('w' if np.all(cs.flag_grids[n, pol]) else 'k')
            ax.text(x, y, str(n), va='center', ha='center', fontsize=8, c=color)
        ax.set_title(pol)
        ax.set_xlabel('East-West Antenna Position (m)')
    
    axes[0].set_ylabel('North-South Antenna Position (m)')

    plt.tight_layout()
    plt.colorbar(im, ax=axes, location='top', aspect=60, pad=.04, label='Average Relative Difference Before and After Smoothing', extend='max')

# *Figure 7: Relative Difference Before and After Smoothing*

Similar to [the above plots](#Figure-6:-Average-χ2-per-Antenna), here we show the relative
difference before and after smoothing, compared to the magnitude of the smoothed calibration
solution. Totally flagged antennas (because they are above the `SC_RELATIVE_DIFF_CUTOFF`) are red in
the first two plots, and their numbers are white in the last plot.

In [ ]:
time_avg_diff_plot()
freq_avg_diff_plot()
avg_difference_array_plot()

## Save Results

In [ ]:
add_to_history = 'Produced by calibration_smoothing notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65

In [ ]:
if SAVE_RESULTS:
    cs.write_smoothed_cal(output_replace=(SKY_CAL_SUFFIX, SMOOTH_CAL_SUFFIX), add_to_history=add_to_history, clobber=True)

In [ ]:
if SAVE_RESULTS:
    # write the day's complete a posteriori flag record: this notebook is the last stage that
    # touches flags, so entirely-flagged times, frequencies, and antennas are all final here
    final_flags = np.all([cs.flag_grids[ant] for ant in cs.ants], axis=0)
    dt = np.median(np.diff(cs.time_grid))
    df = np.median(np.diff(cs.freqs))
    out_yml_str = 'JD_flags: ' + str([[float(cs.time_grid[fs][0] - dt / 2), float(cs.time_grid[fs][-1] + dt / 2)]
                                      for fs in true_stretches(np.all(final_flags, axis=1))])
    out_yml_str += '\n\nfreq_flags: ' + str([[float(cs.freqs[fs][0] - df / 2), float(cs.freqs[fs][-1] + df / 2)]
                                             for fs in true_stretches(np.all(final_flags, axis=0))])
    out_yml_str += '\n\nex_ants: ' + str([[int(ant[0]), ant[1]] for ant in sorted(cs.ants)
                                          if np.all(cs.flag_grids[ant])]).replace("'", "")

    print(f'Writing the following to {out_yaml_file}\n' + '-' * (25 + len(out_yaml_file)))
    print(out_yml_str)
    with open(out_yaml_file, 'w') as outfile:
        outfile.writelines(out_yml_str)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')